# BÀI 2. THỰC HÀNH XÂY DỰNG MÔ HÌNH PHÂN LOẠI CHO DỮ LIỆU WINE QUALITY

Mục tiêu: Xây dựng quy trình phân loại chất lượng rượu vang từ bộ dữ liệu `wine_quality_merged.csv` (6.497 mẫu, 13 cột) theo đầy đủ 5 bước thực hành trong bài giảng: Chuẩn bị dữ liệu, Khám phá dữ liệu (EDA), Xây dựng mô hình & Cross Validation, Xử lý mất cân bằng dữ liệu (Class Imbalance), và Đánh giá mô hình trên tập Test.


# PHẦN 1. CHUẨN BỊ DỮ LIỆU

Nạp thư viện, đọc tệp `wine_quality_merged.csv` và tạo biến mục tiêu nhị phân `target`:
* `target = 1`: Rượu chất lượng cao (`quality >= 7`).
* `target = 0`: Rượu chất lượng thường (`quality < 7`).


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate
)
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
    ConfusionMatrixDisplay,
    PrecisionRecallDisplay,
    make_scorer
)

from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline

RANDOM_STATE = 42
pd.set_option("display.max_columns", None)

df = pd.read_csv("wine_quality_merged.csv")
df["target"] = (df["quality"] >= 7).astype(int)

print("Kích thước dữ liệu gốc:", df.shape)
display(df.head())


# PHẦN 2. KHÁM PHÁ DỮ LIỆU (EDA)

## Bước 1. Kiểm tra cấu trúc, giá trị thiếu và kiểu dữ liệu


In [ ]:
print("Shape:", df.shape)
print("\nSố giá trị thiếu:")
print(df.isna().sum().sum())

print("\nSố dòng trùng lặp:")
print(df.duplicated().sum())

print("\nKiểu dữ liệu:")
display(df.dtypes.value_counts())


## Bước 2. Phân tích phân bố lớp và trực quan hóa


In [ ]:
class_count = df["target"].value_counts().sort_index()
class_percent = df["target"].value_counts(normalize=True).sort_index() * 100

class_distribution = pd.DataFrame({
    "Số mẫu": class_count,
    "Tỷ lệ (%)": class_percent.round(2)
})
class_distribution.index = ["Thường (0: quality < 7)", "Chất lượng cao (1: quality >= 7)"]
display(class_distribution)

plt.figure(figsize=(6, 4))
class_count.plot(kind="bar", color=["#4C72B0", "#DD8452"])
plt.title("Phân bố hai lớp chất lượng rượu")
plt.xlabel("Target")
plt.ylabel("Số mẫu")
plt.xticks(
    ticks=[0, 1],
    labels=["Thường (0)", "Chất lượng cao (1)"],
    rotation=0
)
plt.tight_layout()
plt.show()


## Bước 3. Kiểm tra thang đo của các đặc trưng số


In [ ]:
num_features = df.drop(columns=["quality", "target", "type"])
feature_scale = pd.DataFrame({
    "mean": num_features.mean(),
    "std": num_features.std(),
    "min": num_features.min(),
    "max": num_features.max()
}).sort_values("std", ascending=False)

display(feature_scale)


**Nhận xét EDA:**
* Bộ dữ liệu gồm **6.497 mẫu** và **13 cột**, không có giá trị bị thiếu (`NaN = 0`).
* **Mất cân bằng lớp (Class Imbalance):** Nhóm rượu chất lượng cao (`target = 1`) chỉ chiếm **19,66%** (1.277 mẫu), trong khi nhóm thường (`target = 0`) chiếm **80,34%** (5.220 mẫu). Do đó không thể chỉ dùng `Accuracy` mà phải ưu tiên theo dõi `F1-score` và `PR-AUC`.
* **Chênh lệch thang đo & biến phân loại:** Cột `total sulfur dioxide` có độ lệch chuẩn ~56,5 (giá trị lên tới 440) trong khi `density` hay `chlorides` nhỏ hơn 1; ngoài ra cột `type` (`red`/`white`) là dạng chữ nên cần dùng `ColumnTransformer` kết hợp `StandardScaler` và `OneHotEncoder`.


# PHẦN 3. XÂY DỰNG MÔ HÌNH VÀ CROSS VALIDATION

## Bước 1. Tách tập Train/Test theo tỷ lệ 80:20 (`stratify=y`)


In [ ]:
X = df.drop(columns=["quality", "target"])
y = df["target"]

num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = X.select_dtypes(include=["object"]).columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)


## Bước 2. Thiết lập bộ tiền xử lý (`ColumnTransformer`) và Cross Validation cơ sở


In [ ]:
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(drop="first"), cat_cols)
])

scoring = {
    "accuracy": "accuracy",
    "precision": make_scorer(precision_score, zero_division=0),
    "recall": make_scorer(recall_score, zero_division=0),
    "f1": make_scorer(f1_score, zero_division=0),
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision"
}

def cv_summary(model, X_data, y_data, cv):
    scores = cross_validate(
        model,
        X_data,
        y_data,
        cv=cv,
        scoring=scoring
    )
    row = {}
    for metric in scoring.keys():
        values = scores[f"test_{metric}"]
        row[f"{metric}_mean"] = values.mean()
        row[f"{metric}_std"] = values.std()
    return row

stratified_kfold = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

baseline_pipeline = Pipeline([
    ("prep", preprocessor),
    ("model", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE))
])

baseline_cv = cross_validate(
    baseline_pipeline,
    X_train,
    y_train,
    cv=stratified_kfold,
    scoring=scoring
)

baseline_cv_table = pd.DataFrame({
    metric: baseline_cv[f"test_{metric}"]
    for metric in scoring.keys()
})

baseline_cv_result = pd.DataFrame({
    "Mean": baseline_cv_table.mean(),
    "Std": baseline_cv_table.std()
})

display(baseline_cv_result.round(4))


# PHẦN 4. XỬ LÝ CLASS IMBALANCE VÀ SO SÁNH CÁC MÔ HÌNH

Thử nghiệm kết hợp các kỹ thuật lấy mẫu (`RandomOverSampler`, `RandomUnderSampler`, `SMOTE`) bên trong `ImbPipeline` cùng các thuật toán phân loại (`Logistic Regression`, `KNN`, `SVM`, `Decision Tree`, `Random Forest`) trên 5-Fold Stratified Cross Validation.


In [ ]:
models = {
    "Logistic Regression (No Sampling)": ImbPipeline([
        ("prep", preprocessor),
        ("model", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE))
    ]),
    "Logistic Regression + ROS": ImbPipeline([
        ("prep", preprocessor),
        ("sampler", RandomOverSampler(random_state=RANDOM_STATE)),
        ("model", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE))
    ]),
    "Logistic Regression + RUS": ImbPipeline([
        ("prep", preprocessor),
        ("sampler", RandomUnderSampler(random_state=RANDOM_STATE)),
        ("model", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE))
    ]),
    "Logistic Regression + SMOTE": ImbPipeline([
        ("prep", preprocessor),
        ("sampler", SMOTE(random_state=RANDOM_STATE)),
        ("model", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE))
    ]),
    "KNN + SMOTE": ImbPipeline([
        ("prep", preprocessor),
        ("sampler", SMOTE(random_state=RANDOM_STATE)),
        ("model", KNeighborsClassifier())
    ]),
    "SVM + SMOTE": ImbPipeline([
        ("prep", preprocessor),
        ("sampler", SMOTE(random_state=RANDOM_STATE)),
        ("model", SVC(probability=True, random_state=RANDOM_STATE))
    ]),
    "Decision Tree + SMOTE": ImbPipeline([
        ("prep", preprocessor),
        ("sampler", SMOTE(random_state=RANDOM_STATE)),
        ("model", DecisionTreeClassifier(random_state=RANDOM_STATE))
    ]),
    "Random Forest (No Sampling)": ImbPipeline([
        ("prep", preprocessor),
        ("model", RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE))
    ]),
    "Random Forest + SMOTE": ImbPipeline([
        ("prep", preprocessor),
        ("sampler", SMOTE(random_state=RANDOM_STATE)),
        ("model", RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE))
    ])
}

cv_rows = []
for name, pipe in models.items():
    res = cv_summary(pipe, X_train, y_train, stratified_kfold)
    res["Model"] = name
    cv_rows.append(res)

cv_comparison = pd.DataFrame(cv_rows).set_index("Model")
comparison_short = cv_comparison[[
    "accuracy_mean",
    "precision_mean",
    "recall_mean",
    "f1_mean",
    "roc_auc_mean",
    "pr_auc_mean"
]].sort_values("f1_mean", ascending=False)

display(comparison_short.round(4))


# PHẦN 5. ĐÁNH GIÁ MÔ HÌNH TỐT NHẤT TRÊN TẬP TEST

Huấn luyện mô hình có `F1-score` và `PR-AUC` cao nhất trên toàn bộ tập `X_train`, sau đó đánh giá khách quan trên tập `X_test` thông qua `Classification Report`, `Confusion Matrix` và `Precision-Recall Curve`.


In [ ]:
best_name = comparison_short.index[0]
best_model = models[best_name]
print("Mô hình tốt nhất được chọn:", best_name)

best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

print("\nTest Accuracy :", round(accuracy_score(y_test, y_pred), 4))
print("Test Precision:", round(precision_score(y_test, y_pred), 4))
print("Test Recall   :", round(recall_score(y_test, y_pred), 4))
print("Test F1-score :", round(f1_score(y_test, y_pred), 4))
print("Test ROC-AUC  :", round(roc_auc_score(y_test, y_prob), 4))
print("Test PR-AUC   :", round(average_precision_score(y_test, y_prob), 4))

print("\nClassification Report trên tập Test:")
print(classification_report(
    y_test,
    y_pred,
    target_names=["Thường (0)", "Chất lượng cao (1)"],
    digits=4,
    zero_division=0
))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    display_labels=["Thường (0)", "Chất lượng cao (1)"],
    ax=axes[0],
    colorbar=False
)
axes[0].set_title(f"Confusion Matrix - {best_name}")

PrecisionRecallDisplay.from_predictions(
    y_test,
    y_prob,
    ax=axes[1]
)
axes[1].set_title(f"Precision-Recall Curve - {best_name}")

plt.tight_layout()
plt.show()


**Kết luận chung:**
* Mô hình cơ sở `Logistic Regression` (không sampling) đạt `Accuracy` ~81,64% nhưng `Recall` rất thấp (~26,83%) do bị ảnh hưởng bởi chênh lệch tỷ lệ lớp (80% vs 20%).
* Khi áp dụng các kỹ thuật sampling (`ROS`, `RUS`, `SMOTE`), chỉ số `Recall` và `F1-score` của nhóm mô hình tuyến tính được cải thiện rõ rệt (từ 0,36 lên >0,51).
* Nhóm mô hình phi tuyến kết hợp cây quyết định (`Random Forest`) vượt trội hơn hẳn nhờ nắm bắt được các tương tác phức tạp giữa các chỉ số hóa học của rượu, đạt **ROC-AUC > 0,91** và **PR-AUC > 0,77** trên cả Cross Validation lẫn tập kiểm thử độc lập `Test`.
